In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import time
import optuna

In [ ]:
# ====================== PRE-COMPUTE MLE (vectorized) ======================
def compute_mu_sigma(closes, M):
    n = len(closes)
    if n < M + 1:
        return np.zeros(n), np.zeros(n)

    logrets = np.log(closes[1:] / closes[:-1])
    logrets_s = pd.Series(logrets)

    roll_alpha = logrets_s.rolling(M, min_periods=M).mean()
    roll_beta2 = logrets_s.rolling(M, min_periods=M).var(ddof=1)

    mu_hat = np.zeros(n)
    sigma_hat = np.full(n, 1e-8)

    num_valid = n - M
    if num_valid > 0:
        alpha_vals = roll_alpha.iloc[M-1 : M-1 + num_valid].values
        beta2_vals = roll_beta2.iloc[M-1 : M-1 + num_valid].values
        mu_hat[M:] = alpha_vals + 0.5 * beta2_vals
        sigma_hat[M:] = np.sqrt(np.maximum(beta2_vals, 0.0))

    return mu_hat, sigma_hat


# ====================== SIMULATION (exact PDF pseudocode) ======================
def simulate(closes, mu_hat, sigma_hat, num_trades_arr, L, c, initial_capital):
    n = len(closes)
    equity = np.zeros(n)
    equity[0] = initial_capital
    X = initial_capital
    f_prev = 0.0
    trades = 0

    for k in range(1, n):
        # Step 1: mark-to-market (pre-decision)
        ret = closes[k] / closes[k-1]
        X = X * (f_prev * ret + (1.0 - f_prev))

        # Step 2-3: estimates + f*
        mu = mu_hat[k]
        sigma = sigma_hat[k]
        if sigma < 1e-8 or mu <= 0:
            f_star = 0.0
        else:
            f_star = min(1.0, mu / (sigma ** 2))

        # Step 4: liquidity gate using real column
        liquid = (num_trades_arr[k] >= L)

        if not liquid:
            f_k = f_prev
            trade_flag = False
        else:
            # Step 5: net-growth test
            g_keep = f_prev * mu - 0.5 * f_prev**2 * sigma**2
            g_trade = f_star * mu - 0.5 * f_star**2 * sigma**2 - c * abs(f_star - f_prev)

            if g_trade > g_keep:
                f_k = f_star
                trade_flag = True
            else:
                f_k = f_prev
                trade_flag = False

        # Step 6: apply cost + update wealth
        cost_factor = 1.0 - c * abs(f_k - f_prev) if trade_flag and liquid else 1.0
        X *= cost_factor

        if trade_flag and liquid and abs(f_k - f_prev) > 1e-8:
            trades += 1

        equity[k] = X
        f_prev = f_k

    return equity, trades


def compute_metrics(equity: np.ndarray, initial_capital: float, n: int):
    final_pnl = equity[-1] - initial_capital
    if n < 2:
        return {'final_pnl': final_pnl, 'sharpe': 0.0, 'ann_sharpe': 0.0, 'max_dd_pct': 0.0}

    returns = np.diff(equity) / equity[:-1]
    sharpe = 0.0
    ann_sharpe = 0.0
    if len(returns) > 30 and np.std(returns) > 1e-8:
        sharpe = np.mean(returns) / np.std(returns)
        ann_sharpe = sharpe * np.sqrt(365.25 * 24 * 60)

    cummax = np.maximum.accumulate(equity)
    dd = (equity - cummax) / cummax
    max_dd_pct = -dd.min() * 100 if dd.min() < 0 else 0.0

    return {
        'final_pnl': final_pnl,
        'sharpe': sharpe,
        'ann_sharpe': ann_sharpe,
        'max_dd_pct': max_dd_pct
    }

In [ ]:
csv_path = "data/DOGEUSDT-1m-2023-01-01_2026-02-28.csv"   # ←←← UPDATE TO YOUR FILE

print("Loading CSV...")
df = pd.read_csv(csv_path)
df['datetime'] = pd.to_datetime(df["OpenTimestamp[ms]"], unit="ms")
df = df.sort_values("datetime").reset_index(drop=True)
n = len(df)
print(f"Data loaded: {n:,} minutes (~{n/(60*24):.1f} days)")

# Real liquidity column from CSV
num_trades_arr = df["NumberOfTrades"].values.astype(float)

closes = df['Close'].values
initial_capital = 1_000_000.0
c = 0.001
M = 41
L = 30

# ====================== DEFAULT BACKTEST ======================
print(f"\nRunning default (M={M}, L={L})...")
mu_def, sigma_def = compute_mu_sigma(closes, M)
equity_before, trades_before = simulate(closes, mu_def, sigma_def, num_trades_arr, L, 0.0, initial_capital)
equity_after, trades_after = simulate(closes, mu_def, sigma_def, num_trades_arr, L, c, initial_capital)
metrics_before = compute_metrics(equity_before, initial_capital, n)
metrics_after = compute_metrics(equity_after, initial_capital, n)

print("\n" + "="*80)
print(f"DEFAULT RESULTS (M={M}, L={L})")
print("="*80)
print(f"Final PnL BEFORE         : ${metrics_before['final_pnl']:,.2f}")
print(f"Final PnL AFTER          : ${metrics_after['final_pnl']:,.2f}")
print(f"Sharpe Ratio BEFORE      : {metrics_before['sharpe']:.4f}")
print(f"Sharpe Ratio AFTER       : {metrics_after['sharpe']:.4f}")
print(f"Annualized Sharpe BEFORE : {metrics_before['ann_sharpe']:.2f}")
print(f"Annualized Sharpe AFTER  : {metrics_after['ann_sharpe']:.2f}")
print(f"Max Drawdown BEFORE      : {metrics_before['max_dd_pct']:.2f}%")
print(f"Max Drawdown AFTER       : {metrics_after['max_dd_pct']:.2f}%")
print(f"Total trades BEFORE      : {trades_before:,}")
print(f"Total trades AFTER       : {trades_after:,}")
print("="*80)

equity_df = pd.DataFrame({'datetime': df['datetime'], 'pnl_before': equity_before - initial_capital, 'pnl_after': equity_after - initial_capital})
equity_df.to_csv('equity_curve.csv', index=False)
print("Saved 'equity_curve.csv' for the Flask dashboard.")

# Cumulative PnL plot (static)
plt.figure(figsize=(14, 7))
plt.plot(df['datetime'], equity_before - initial_capital, label='Before 0.1% cost', color='royalblue')
plt.plot(df['datetime'], equity_after - initial_capital, label='After 0.1% cost', color='crimson')
plt.title(f'Strategy Backtest (M={M}, L={L})', fontsize=16)
plt.xlabel('Time')
plt.ylabel('Cumulative PnL (USD)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# ====================== OPTUNA OBJECTIVE ======================
def objective(trial, closes, num_trades_arr, initial_capital, n, c=0.001):
    M = trial.suggest_int('M', 30, 300)
    L = trial.suggest_int('L', 5, 50)

    mu_hat, sigma_hat = compute_mu_sigma(closes, M)
    equity, trades = simulate(closes, mu_hat, sigma_hat, num_trades_arr, L, c, initial_capital)
    metrics = compute_metrics(equity, initial_capital, n)
    return metrics['ann_sharpe']

# ====================== OPTUNA OPTIMIZATION ======================
n_trials = 1_000

print(f"\nStarting Optuna ({n_trials} trials) — tuning M and L...")
start_time = time.time()

study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=67))
study.optimize(lambda trial: objective(trial, closes, num_trades_arr, initial_capital, n, c),
               n_trials=n_trials, show_progress_bar=True)

best = study.best_params
best_sharpe = study.best_value
print(f"\nOPTUNA BEST: M={best['M']}, L={best['L']} → Sharpe = {best_sharpe:.4f}")
print(f"Finished in {time.time() - start_time:.1f}s")

# ====================== RUN BEST & SAVE ======================
print("\nRunning best parameters...")
mu_best, sigma_best = compute_mu_sigma(closes, best['M'])
equity_best_after, trades_best_after = simulate(closes, mu_best, sigma_best, num_trades_arr, best['L'], c, initial_capital)
metrics_best = compute_metrics(equity_best_after, initial_capital, n)

equity_df = pd.DataFrame({
    'datetime': df['datetime'],
    'pnl_before': equity_best_after - initial_capital,
    'pnl_after': equity_best_after - initial_capital
})
equity_df.to_csv('equity_curve.csv', index=False)
print("✅ Saved 'equity_curve.csv' (BEST parameters)")

print("\n" + "="*80)
print(f"BEST RESULTS (M={best['M']}, L={best['L']})")
print("="*80)
print(f"Final PnL         : ${metrics_best['final_pnl']:,.2f}")
print(f"Sharpe Ratio      : {metrics_best['sharpe']:.4f}")
print(f"Ann. Sharpe       : {metrics_best['ann_sharpe']:.2f}")
print(f"Max DD            : {metrics_best['max_dd_pct']:.2f}%")
print(f"Total trades      : {trades_best_after:,}")
print("="*80)

plt.figure(figsize=(14, 7))
plt.plot(df['datetime'], equity_best_after - initial_capital, color='royalblue', linewidth=1.5)
plt.title(f'BEST GBM-Kelly Strategy (M={best["M"]}, L={best["L"]})', fontsize=16)
plt.xlabel('Time')
plt.ylabel('Cumulative PnL (USD)')
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()